# Stage 1 -> 2 — Per-image features

Turn the segmentation masks into quantitative, image-level features:
- `lesion_area_fraction` — rust pixels / leaf pixels
- `blob_count` — number of distinct lesions
- `blob_size_mean/std/max` — lesion size distribution
- `blob_density` — lesions per 10k leaf pixels

In [ ]:
from _setup import DATA_DIR, ARTIFACTS_DIR, ensure_dataset
import numpy as np
import matplotlib.pyplot as plt
from phytolabs import segmentation, pipeline, viz
from phytolabs.features import FEATURE_NAMES

data_dir = ensure_dataset()
gmm_path = ARTIFACTS_DIR / 'gmm.joblib'
if gmm_path.exists():
    leaf_gmm = segmentation.LeafGMM.load(gmm_path)
else:
    leaf_gmm = pipeline.fit_gmm_from_dir(data_dir / 'train')
    leaf_gmm.save(gmm_path)

## Build a feature table for the training split

In [ ]:
X, y, paths = pipeline.build_feature_table(data_dir / 'train', leaf_gmm)
print('X shape:', X.shape, '| positives (rust):', int(y.sum()))
import numpy as np
for name, col in zip(FEATURE_NAMES, X.T):
    print(f'{name:22s} healthy_mean={col[y==0].mean():.3f}  rust_mean={col[y==1].mean():.3f}')

## Feature distributions by class

Good features separate healthy (blue) from rust (orange).

In [ ]:
viz.plot_feature_histograms(X, y, FEATURE_NAMES)
plt.show()

## Save feature tables (train + val) for Stage 2

In [ ]:
for split in ('train', 'val'):
    Xs, ys, ps = pipeline.build_feature_table(data_dir / split, leaf_gmm)
    out = ARTIFACTS_DIR / f'features_{split}.npz'
    np.savez(out, X=Xs, y=ys, paths=np.array(ps, dtype=object))
    print(f'{split}: {Xs.shape} -> {out}')